In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [2]:
# CREATE TRAINING AND TESTING SETS

# Divide subjects into training and testing sets

SUBJECTS = helper_functions.fuglsang_get_subjects()
test_percent = 0.3
num_testing_subjects = int(len(SUBJECTS)*(test_percent))
print(num_testing_subjects)
training_subjects = SUBJECTS[:-num_testing_subjects]
testing_subjects = SUBJECTS[len(SUBJECTS)-num_testing_subjects::]

# -----------------------------------------------------------------------------------------------------
# Load backward TRFs for all subjects

trf_types = [
    # Single predictors
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False),
]

training_trf_data, n_training_subjects = helper_functions.load_trfs(DATASET_TYPE.FUGLSANG, training_subjects, trf_types, trf_dir=FUGLSANG_TRF_DIR)
#testing_trf_data, n_testing_subjects = helper_functions.load_trfs(testing_subjects, trf_types, trf_dir=FUGLSANG_TRF_DIR)

print('training and validation sets complete')

5
  ✓ S1
  ✓ S2
  ✓ S3
  ✓ S4
  ✓ S5
  ✓ S6
  ✓ S7
  ✓ S8
  ✓ S9
  ✓ S10
  ✓ S11
  ✓ S12
  ✓ S13

Loaded: 13 subjects | Skipped: 0 subjects
training and validation sets complete


In [6]:
# MAKE UNIVERSAL TRFS USING TRAINING DATA

# Extract model names
models = []
for type in trf_types:
    models.append(helper_functions.get_trf_model_name(DATASET_TYPE.FUGLSANG, *type))

# -----------------------------------------------------------------------------------------------------
# Loop over all models
for model in models:
    print(f'\nProcessing universal TRF for model: {model}')

    # Collect TRFs for all subjects for this model
    training_trf_list = [trf.h_scaled for trf in training_trf_data[model]]
    print(training_trf_list)

    # Combine and average across subjects
    universal_trf = eelbrain.combine(training_trf_list).mean('case')
    print(universal_trf)

    # Save universal TRF for this model
    eelbrain.save.pickle(universal_trf, FUGLSANG_TRF_DIR / f'universal-{model}.pickle')

    print(f'Saved universal TRF for model {model}')


Processing universal TRF for model: backward_attended_envelope
[<NDVar 'S1_eeg': 64 sensor, 73 time>, <NDVar 'S2_eeg': 64 sensor, 73 time>, <NDVar 'S3_eeg': 64 sensor, 73 time>, <NDVar 'S4_eeg': 64 sensor, 73 time>, <NDVar 'S5_eeg': 64 sensor, 73 time>, <NDVar 'S6_eeg': 64 sensor, 73 time>, <NDVar 'S7_eeg': 64 sensor, 73 time>, <NDVar 'S8_eeg': 64 sensor, 73 time>, <NDVar 'S9_eeg': 64 sensor, 73 time>, <NDVar 'S10_eeg': 64 sensor, 73 time>, <NDVar 'S11_eeg': 64 sensor, 73 time>, <NDVar 'S12_eeg': 64 sensor, 73 time>, <NDVar 'S13_eeg': 64 sensor, 73 time>]
<NDVar 'S': 64 sensor, 73 time>
Saved universal TRF for model backward_attended_envelope

Processing universal TRF for model: backward_ignored_envelope
[<NDVar 'S1_eeg': 64 sensor, 73 time>, <NDVar 'S2_eeg': 64 sensor, 73 time>, <NDVar 'S3_eeg': 64 sensor, 73 time>, <NDVar 'S4_eeg': 64 sensor, 73 time>, <NDVar 'S5_eeg': 64 sensor, 73 time>, <NDVar 'S6_eeg': 64 sensor, 73 time>, <NDVar 'S7_eeg': 64 sensor, 73 time>, <NDVar 'S8_eeg': 6

In [4]:
# MAKE PREDICTED ENVELOPES AND COMPUTE CORRELATION

r_values = {}
r2_values = {}

for model in models:
    r_values[model] = []
    r2_values[model] = []

for subject in testing_subjects:
    # Get their EEG
    eeg = eelbrain.load.unpickle(FUGLSANG_EEG_DIR / subject / f'{subject}_eeg.pickle')

    # Predict envelopes
    for model in models:
        trf = eelbrain.load.unpickle(FUGLSANG_TRF_DIR / f'universal-{model}.pickle')
        predictor_name = model.split('backward_')[1]
        predictor = eelbrain.load.unpickle(FUGLSANG_PREDICTOR_DIR / subject / f'{predictor_name}_concat.pickle')
        predicted_envelope = eelbrain.convolve(trf, eeg)

        # Convert to numpy arrays
        env = predictor.x
        pred = predicted_envelope.x

        # Correlation and r²
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values[model].append(r)
        r2_values[model].append(r2)

        print(f'Correlation values for model: {model}')
        print(f'r = {r:.4f}, r² = {r2:.4f}')

print('\n===== Universal decoder summary =====')

for model in models:
    mean_r = np.mean(r_values[model])
    std_r = np.std(r_values[model])
    mean_r2 = np.mean(r2_values[model])
    std_r2 = np.std(r2_values[model])

    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, '
        f'r² = {mean_r2:.4f} ± {std_r2:.4f}, ')

Correlation values for model: backward_attended_envelope
r = 0.1748, r² = 0.0305
Correlation values for model: backward_ignored_envelope
r = 0.0679, r² = 0.0046
Correlation values for model: backward_attended_envelope_onset
r = 0.0766, r² = 0.0059
Correlation values for model: backward_ignored_envelope_onset
r = 0.0330, r² = 0.0011
Correlation values for model: backward_attended_envelope
r = 0.2021, r² = 0.0408
Correlation values for model: backward_ignored_envelope
r = 0.0766, r² = 0.0059
Correlation values for model: backward_attended_envelope_onset
r = 0.0874, r² = 0.0076
Correlation values for model: backward_ignored_envelope_onset
r = 0.0350, r² = 0.0012
Correlation values for model: backward_attended_envelope
r = 0.0956, r² = 0.0091
Correlation values for model: backward_ignored_envelope
r = 0.0863, r² = 0.0074
Correlation values for model: backward_attended_envelope_onset
r = 0.0511, r² = 0.0026
Correlation values for model: backward_ignored_envelope_onset
r = 0.0338, r² = 0.001

In [5]:
# CLASSIFICATION

def aad_classifier_universal(predictor, attention_att, attention_ign):

    att_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor, attention_att, MODEL_TYPE.BACKWARD, False
    )
    ign_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor, attention_ign, MODEL_TYPE.BACKWARD, False
    )

    # Load universal TRFs
    trf_att = eelbrain.load.unpickle(FUGLSANG_TRF_DIR / f'universal-{att_name}.pickle')
    trf_ign = eelbrain.load.unpickle(FUGLSANG_TRF_DIR / f'universal-{ign_name}.pickle')

    correct = 0
    total   = 0

    for subject in testing_subjects:

        # ----------------------------------------
        # Load EEG
        # ----------------------------------------
        eeg = eelbrain.load.unpickle(
            FUGLSANG_EEG_DIR / subject / f'{subject}_eeg.pickle'
        )

        # ----------------------------------------
        # Load true stimuli
        # ----------------------------------------
        att_stim = eelbrain.load.unpickle(
            FUGLSANG_PREDICTOR_DIR / subject / f'{att_name.split("backward_")[1]}_concat.pickle'
        )

        ign_stim = eelbrain.load.unpickle(
            FUGLSANG_PREDICTOR_DIR / subject / f'{ign_name.split("backward_")[1]}_concat.pickle'
        )

        # ----------------------------------------
        # Reconstruct envelopes
        # ----------------------------------------
        pred_att = eelbrain.convolve(trf_att, eeg).x
        pred_ign = eelbrain.convolve(trf_ign, eeg).x

        att_env = att_stim.x
        ign_env = ign_stim.x

        # ----------------------------------------
        # Compute correlations
        # ----------------------------------------
        r_att_att = np.corrcoef(pred_att, att_env)[0,1]
        r_att_ign = np.corrcoef(pred_att, ign_env)[0,1]

        r_ign_att = np.corrcoef(pred_ign, att_env)[0,1]
        r_ign_ign = np.corrcoef(pred_ign, ign_env)[0,1]

        # ----------------------------------------
        # Decision rule
        # ----------------------------------------
        # attended TRF should match attended stimulus better
        correct += (r_att_att > r_att_ign)
        total += 1

        print(f"{subject}: r_att_att={r_att_att:.3f}, r_att_ign={r_att_ign:.3f}")

    acc = correct / total if total > 0 else 0

    print(f"\n✅ Classification rate ({att_name} vs {ign_name}): {acc:.2%}")
    print('\n————————————————————————————————————————————————————————-\n')

    return acc


# Envelope backward model
aad_classifier_universal(PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.ATTENDED, ATTENTION_TYPE.IGNORED)

# Envelope onset backward model
aad_classifier_universal(PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, ATTENTION_TYPE.IGNORED)


S14: r_att_att=0.175, r_att_ign=0.044
S15: r_att_att=0.202, r_att_ign=0.057
S16: r_att_att=0.096, r_att_ign=0.085
S17: r_att_att=0.088, r_att_ign=0.026
S18: r_att_att=0.148, r_att_ign=0.021

✅ Classification rate (backward_attended_envelope vs backward_ignored_envelope): 100.00%

————————————————————————————————————————————————————————-

S14: r_att_att=0.077, r_att_ign=0.025
S15: r_att_att=0.087, r_att_ign=0.024
S16: r_att_att=0.051, r_att_ign=0.031
S17: r_att_att=0.035, r_att_ign=0.003
S18: r_att_att=0.059, r_att_ign=0.016

✅ Classification rate (backward_attended_envelope_onset vs backward_ignored_envelope_onset): 100.00%

————————————————————————————————————————————————————————-



np.float64(1.0)